# Rotating strong-wind LES example

This notebook compares the official LES endpoints with OceanTurb CATKE, KPP, MY2.5, $k$-$\omega$, and $k$-$\epsilon$. Every OceanTurb integration starts from the same regridded LES $u$, $v$, buoyancy, and passive-tracer profiles at $t=600$ s and uses a common 10 s time step.

The LES wind cases include wave/Stokes effects. The present OceanTurb columns do not include an explicit Langmuir/Stokes-drift enhancement, so this is a controlled physical-case comparison rather than an exact reproduction of every process in the reference LES.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

experiment_dir = Path.cwd()
output_dir = experiment_dir / "output" / "dt10s"
figure_dir = experiment_dir / "figures"
figure_dir.mkdir(exist_ok=True)

suites = [
    (6,  r"$\tau_x=-1.4\times10^{-3}$ m$^2$ s$^{-2}$"),
    (12, r"$\tau_x=-9.0\times10^{-4}$ m$^2$ s$^{-2}$"),
    (24, r"$\tau_x=-6.8\times10^{-4}$ m$^2$ s$^{-2}$"),
    (48, r"$\tau_x=-4.5\times10^{-4}$ m$^2$ s$^{-2}$"),
    (72, r"$\tau_x=-4.1\times10^{-4}$ m$^2$ s$^{-2}$"),
]
profiles = {
    hours: pd.read_csv(output_dir / f"strong_wind_{hours:02d}h.csv")
    for hours, _ in suites
}

styles = [
    ("catke", "CATKE", "#111111", "-", 2.0),
    ("kpp", "KPP", "#2878B5", "--", 2.0),
    ("my25", "MY2.5", "#E09F28", "-.", 2.0),
    ("komega", r"$k$-$\omega$", "#D64B3C", ":", 2.2),
    ("kepsilon", r"$k$-$\epsilon$", "#8055A6", (0, (6, 2)), 2.0),
]

list(profiles[6].columns)


In [ ]:
def finish_axis(ax, column, y_limits, xlabel):
    ax.set_ylim(*y_limits)
    ax.set_xlabel(xlabel)
    ax.margins(x=0.07)
    ax.grid(alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if column == 0:
        ax.set_ylabel("z (m)")
    else:
        ax.tick_params(labelleft=False)

def draw_scalar_profiles(y_limits, filename, title_suffix):
    fig, axes = plt.subplots(2, 5, figsize=(17.0, 8.0), sharey=False)

    for column, (hours, stress_label) in enumerate(suites):
        data = profiles[hours]
        z = data["z_m"]
        b_reference = data.loc[27, "initial_les_buoyancy_m_s-2"]
        ax_b, ax_c = axes[0, column], axes[1, column]

        ax_b.plot(
            1e4 * (data["final_les_buoyancy_m_s-2"] - b_reference), z,
            color="seagreen", alpha=0.55, linewidth=6.0,
            label="LES", solid_capstyle="round",
        )
        ax_c.plot(
            data["final_les_passive_tracer"], z,
            color="seagreen", alpha=0.55, linewidth=6.0,
            label="LES", solid_capstyle="round",
        )

        for key, label, color, linestyle, linewidth in styles:
            ax_b.plot(
                1e4 * (data[f"{key}_buoyancy_m_s-2"] - b_reference), z,
                color=color, linestyle=linestyle, linewidth=linewidth,
                label=label,
            )
            ax_c.plot(
                data[f"{key}_passive_tracer"], z,
                color=color, linestyle=linestyle, linewidth=linewidth,
            )

        ax_b.set_title(f"{hours} hour simulation\n{stress_label}", fontsize=10.5)
        panel_limits = y_limits[column] if isinstance(y_limits, list) else y_limits
        finish_axis(ax_b, column, panel_limits, "Buoyancy\n($10^{-4}$ m s$^{-2}$)")
        finish_axis(ax_c, column, panel_limits, "Passive tracer")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=6, frameon=False,
        bbox_to_anchor=(0.5, 0.925),
    )
    fig.suptitle(
        "Rotating strong-wind mixing: OceanTurb closures compared with LES"
        f"\nCommon initialization at 10 minutes and $\\Delta t=10$ s{title_suffix}",
        y=0.985, fontsize=15,
    )
    fig.subplots_adjust(
        left=0.055, right=0.985, bottom=0.07, top=0.84,
        wspace=0.18, hspace=0.38,
    )
    path = figure_dir / filename
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.close(fig)
    return path

full_depth_path = draw_scalar_profiles(
    (-256, 0),
    "rotating_strong_wind_closures_full_depth.png",
    " — full depth",
)
full_depth_path


In [ ]:
# An upper-ocean vertical zoom; horizontal limits remain data-driven.
vertical_zoom_path = draw_scalar_profiles(
    [(-175, 5), (-175, 5), (-165, 5), (-155, 5), (-155, 5)],
    "rotating_strong_wind_closures.png",
    " — upper-ocean vertical zoom",
)
vertical_zoom_path


In [ ]:
def draw_velocity_profiles():
    fig, axes = plt.subplots(2, 5, figsize=(17.0, 8.0), sharey=False)
    for column, (hours, stress_label) in enumerate(suites):
        data = profiles[hours]
        z = data["z_m"]
        ax_u, ax_v = axes[0, column], axes[1, column]

        for ax, component in ((ax_u, "u"), (ax_v, "v")):
            ax.plot(
                data[f"final_les_{component}_m_s-1"], z,
                color="seagreen", alpha=0.55, linewidth=6.0,
                label="LES", solid_capstyle="round",
            )
            for key, label, color, linestyle, linewidth in styles:
                ax.plot(
                    data[f"{key}_{component}_m_s-1"], z,
                    color=color, linestyle=linestyle, linewidth=linewidth,
                    label=label,
                )

        ax_u.set_title(f"{hours} hour simulation\n{stress_label}", fontsize=10.5)
        finish_axis(ax_u, column, (-256, 0), "$u$ (m s$^{-1}$)")
        finish_axis(ax_v, column, (-256, 0), "$v$ (m s$^{-1}$)")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=6, frameon=False,
        bbox_to_anchor=(0.5, 0.925),
    )
    fig.suptitle(
        "Rotating strong-wind velocity profiles: official LES and OceanTurb"
        "\ncommon initialization at 10 minutes and $\\Delta t=10$ s",
        y=0.985, fontsize=15,
    )
    fig.subplots_adjust(
        left=0.055, right=0.985, bottom=0.07, top=0.84,
        wspace=0.18, hspace=0.38,
    )
    path = figure_dir / "rotating_strong_wind_velocity_profiles.png"
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.close(fig)
    return path

velocity_path = draw_velocity_profiles()
velocity_path


In [ ]:
summary = pd.read_csv(output_dir / "run_summary.txt", skiprows=10)
summary
